# 02 — Yield Trends Across EU Countries (1995–2024)

**Research Question 1:** *How have crop yields evolved in the EU over the last 30 years?*

This notebook produces one line chart per crop, showing annual yield (t/ha) for each of the six countries from 1995 to 2024. Trends are interpreted in the closing summary.


## 1. Imports and configuration

A consistent country colour palette is defined here and reused across all visualisation notebooks (02–04) for a unified portfolio look. Each colour was chosen for high contrast against a white background.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Paths
DATA_DIR = Path("../data")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Plot theme
sns.set_theme(style="whitegrid")

# Country colour palette (shared across notebooks 02–04)
COUNTRY_COLORS = {
    "France":      "#D62828",  # strong red
    "Germany":     "#1D3557",  # oxford blue
    "Italy":       "#1F7A6F",  # dark teal
    "Spain":       "#B8860B",  # dark goldenrod
    "Portugal":    "#6A0572",  # deep purple
    "Netherlands": "#BC4B24",  # rust orange
}


## 2. Load processed data

Reload the joined dataset produced by notebook 01.


In [ ]:
eu_stat_joined = pd.read_csv(DATA_DIR / "processed" / "eu_stat_joined.csv")

print(f"Shape: {eu_stat_joined.shape}")
print(f"Countries: {sorted(eu_stat_joined['Area'].unique())}")
print(f"Crops: {sorted(eu_stat_joined['Item'].unique())}")
print(f"Year range: {eu_stat_joined['Year'].min()} – {eu_stat_joined['Year'].max()}")
eu_stat_joined.head()


## 3. Per-crop plot configuration

Each crop has a different yield magnitude — sugar beet is measured in tens of t/ha while grapes and wheat are in single digits. The `step` value defines the y-axis tick spacing for each crop.


In [ ]:
CROP_SETTINGS = {
    "Grapes":       {"step": 2},
    "Maize (corn)": {"step": 2},
    "Wheat":        {"step": 2},
    "Sugar beet":   {"step": 10},
}


## 4. Plot annual yield trend per crop

One figure per crop. The x-axis spans 1995–2024 with five-year tick intervals to keep labels readable. The y-axis starts at zero so absolute magnitudes are comparable rather than exaggerated.

The `sns.relplot` figure-level function is used with `kind="line"`, then the underlying axes are accessed via `g.ax` for fine-grained tick control. Files are saved at 400 dpi for print-quality output.


In [ ]:
for crop, settings in CROP_SETTINGS.items():
    crop_df = eu_stat_joined[eu_stat_joined["Item"] == crop].copy()

    g = sns.relplot(
        data=crop_df,
        x="Year",
        y="Annual Yield (t/ha)",
        hue="Area",
        palette=COUNTRY_COLORS,
        kind="line",
        dashes=False,
        marker="o",
        markersize=3,
        linewidth=1.5,
        height=6,
        aspect=14 / 6,
    )

    ax = g.ax
    ax.set_facecolor("white")

    # X-axis: 1995–2024, 5-year intervals
    ax.set_xlim(1995, 2024)
    ax.set_xticks(np.arange(1995, 2025, 5))
    ax.tick_params(axis="x", rotation=0)

    # Y-axis: start at zero, crop-specific tick step
    step = settings["step"]
    max_yield = crop_df["Annual Yield (t/ha)"].max()
    ax.set_yticks(np.arange(0, max_yield + step, step))
    ax.set_ylim(bottom=0)

    g.set_axis_labels("Year", "Yield (tonnes per hectare)")
    g.figure.suptitle(f"Annual {crop} Yield Trend", y=1.02)

    sns.move_legend(
        g,
        loc="upper left",
        bbox_to_anchor=(1.05, 1),
        title="Country",
        frameon=True,
    )

    g.figure.tight_layout()

    file_name = (
        crop.lower()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        + "_yield_evolution.png"
    )
    g.figure.savefig(FIGURES_DIR / file_name, dpi=400, bbox_inches="tight")
    plt.show()


## Summary of findings

**Wheat:** Northern producers (Netherlands, Germany, France) maintain consistently higher yields (~7–9 t/ha) than Southern producers (Italy ~3.5 t/ha, Spain ~3 t/ha, Portugal ~2 t/ha). The gap has remained roughly stable over three decades.

**Maize (corn):** A clear convergence story — Spain has overtaken Northern producers since 2010 and now leads at ~12 t/ha, while Netherlands shows high volatility. Portugal has steadily improved from ~5 to ~9 t/ha.

**Grapes:** Germany historically held the highest yields (~14 t/ha) but has trended downward, while Italy and Netherlands have risen. Southern producers (Spain, Portugal) remain at the lower end (~5–7 t/ha) but show modest improvement.

**Sugar beet:** Northern producers dominate with yields of 70–95 t/ha versus Southern yields of 40–65 t/ha. Portugal exited the dataset around 2017 (no recent observations). The gap is structural rather than narrowing.

These per-country trends set up Research Question 3 (regional convergence) and provide the raw input for Research Question 2 (efficiency). Detailed quantitative comparisons follow in the next notebooks.
